In [32]:
# Colab / 로컬 공통 import
from dataclasses import dataclass, asdict, replace
import json
import platform
import importlib.metadata
from pathlib import Path
import zipfile

import numpy as np
import pandas as pd
import soundfile as sf
from scipy import signal
from scipy.ndimage import uniform_filter1d
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
from IPython.display import display


In [33]:
# Google Colab Drive 연결
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Google Colab이 아닙니다. 아래 DATA_ROOT를 로컬 데이터 폴더로 지정하세요.")

BASE_DIR = Path("/content/drive/MyDrive/signals_and_systems")
ZIP_PATH = BASE_DIR / "assets" / "noise_reduction_challenge.zip"

# 현재 환경에서 우선 확인할 데이터 폴더 후보
DATA_CANDIDATES = [
    Path("/content/noise_reduction_challenge"),
    BASE_DIR / "noise_reduction_challenge",
    BASE_DIR / "noise_reduction_challenge" / "noise_reduction_challenge",
    Path("release/noise_reduction_challenge"),
]

def is_dataset_root(path):
    path = Path(path)
    return all((path / split / "samples.csv").is_file()
               for split in ("train", "validation"))

DATA_ROOT = next((p.resolve() for p in DATA_CANDIDATES if is_dataset_root(p)), None)

# 데이터 폴더가 없을 때에만 ZIP 압축 해제
if DATA_ROOT is None and ZIP_PATH.is_file():
    EXTRACT_DIR = Path("/content") if IN_COLAB else Path.cwd()
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)

    possible_roots = [
        EXTRACT_DIR / "noise_reduction_challenge",
        EXTRACT_DIR / "noise_reduction_challenge" / "noise_reduction_challenge",
    ]
    DATA_ROOT = next((p.resolve() for p in possible_roots if is_dataset_root(p)), None)

if DATA_ROOT is None:
    raise FileNotFoundError(
        "데이터 폴더를 찾지 못했습니다. DATA_CANDIDATES 또는 ZIP_PATH를 확인하세요."
    )

print("DATA_ROOT:", DATA_ROOT)
print("Train manifest:", (DATA_ROOT / "train" / "samples.csv").is_file())
print("Validation manifest:", (DATA_ROOT / "validation" / "samples.csv").is_file())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_ROOT: /content/noise_reduction_challenge
Train manifest: True
Validation manifest: True


In [34]:
from dataclasses import dataclass, asdict
import json
import platform
import importlib.metadata

import numpy as np
import pandas as pd
import soundfile as sf
from scipy import signal
from scipy.ndimage import uniform_filter1d
from tqdm.auto import tqdm


@dataclass(frozen=True)
class Config:
    sample_rate: int = 16000
    n_fft: int = 512
    hop: int = 128
    filter_order: int = 4
    lowpass_hz: float = 3500.0
    highpass_hz: float = 100.0
    bandstop_hz: tuple = (950.0, 1050.0)
    basic_stages: tuple = ("highpass", "lowpass", "bandstop")
    noise_fraction: float = 0.20
    subtraction_alpha: float = 1.5
    subtraction_floor: float = 0.02
    wiener_smoothing: float = 0.98
    wiener_gain_floor: float = 0.05

    def __post_init__(self):
        if self.sample_rate != 16000:
            raise ValueError("This baseline uses 16000 Hz / wideband PESQ.")
        if not 0 < self.hop <= self.n_fft or self.n_fft < 4:
            raise ValueError("Require 0 < hop <= n_fft and n_fft >= 4.")
        if not signal.check_NOLA("hann", self.n_fft, self.n_fft - self.hop):
            raise ValueError("STFT window/hop does not satisfy NOLA.")
        if not 0 < self.noise_fraction <= 1:
            raise ValueError("noise_fraction must be in (0, 1].")
        if not 0 <= self.wiener_smoothing < 1:
            raise ValueError("wiener_smoothing must be in [0, 1).")
        if not 0 <= self.subtraction_floor <= 1 or self.subtraction_alpha < 0:
            raise ValueError("Invalid spectral subtraction parameters.")
        if not 0 <= self.wiener_gain_floor <= 1:
            raise ValueError("wiener_gain_floor must be in [0, 1].")
        nyquist = self.sample_rate / 2
        if not 0 < self.highpass_hz < self.lowpass_hz < nyquist:
            raise ValueError("Require 0 < highpass < lowpass < Nyquist.")
        if not 0 < self.bandstop_hz[0] < self.bandstop_hz[1] < nyquist:
            raise ValueError("Invalid band-stop interval.")
        if self.filter_order < 1 or int(self.filter_order) != self.filter_order:
            raise ValueError("filter_order must be a positive integer.")
        if any(s not in ("lowpass", "highpass", "bandstop") for s in self.basic_stages):
            raise ValueError("Unknown basic filter stage.")


CFG = Config()


def waveform(x):
    x = np.asarray(x, dtype=np.float64)
    if x.ndim != 1 or x.size == 0 or not np.isfinite(x).all():
        raise ValueError("Expected a nonempty, finite mono waveform.")
    return x


In [35]:
def resolve_local_data_path(data_path="release/noise_reduction_challenge", start=None):
    """Resolve an explicit local path from the working folder or its parents."""
    data_path = Path(data_path).expanduser()
    start = Path(start or Path.cwd()).resolve()
    candidates = ([data_path] if data_path.is_absolute() else
                  [parent / data_path for parent in (start, *start.parents)])
    # The default also works after extracting the folder outside this workspace.
    if data_path == Path("release/noise_reduction_challenge"):
        for parent in (start, *start.parents):
            candidates.extend((parent, parent / "noise_reduction_challenge"))
    for candidate in candidates:
        if all((candidate / split / "samples.csv").is_file()
               for split in ("train", "validation")):
            return candidate.resolve()
    raise FileNotFoundError(
        f"LOCAL_DATA_PATH={str(data_path)!r} could not be found from {start}. "
        "Set it to the folder containing train and validation, each with samples.csv."
    )




def load_manifest(root, split):
    if split not in ("train", "validation"):
        raise ValueError("split must be train or validation")
    root = Path(root).resolve()
    df = pd.read_csv(root / split / "samples.csv")
    required = {"segment_id", "clean_path", "noisy_path", "sample_rate_hz", "num_samples",
                "noise_category", "noise_collection", "noise_code", "noise_difficulty",
                "target_snr_db", "split"}
    if not required.issubset(df.columns) or df.empty:
        raise ValueError("Empty manifest or required columns missing.")
    if df.segment_id.duplicated().any():
        raise ValueError("Duplicate segment IDs.")
    if "split" in df and not df["split"].eq(split).all():
        raise ValueError("Manifest split mismatch.")
    for column in ("clean_path", "noisy_path"):
        for relative in df[column]:
            path = (root / relative).resolve()
            if not path.is_relative_to(root) or not path.is_file():
                raise FileNotFoundError(f"Invalid/missing file: {path}")
    return df


def read_audio(root, row, kind, cfg=CFG):
    x, sr = sf.read(Path(root) / row[f"{kind}_path"], dtype="float64")
    x = waveform(x)
    if sr != cfg.sample_rate or sr != int(row["sample_rate_hz"]):
        raise ValueError(f"Sample-rate mismatch for {row['segment_id']}: {sr}")
    if x.size != int(row["num_samples"]):
        raise ValueError(f"Length mismatch for {row['segment_id']}")
    return x


def read_pair(root, row, cfg=CFG):
    return read_audio(root, row, "clean", cfg), read_audio(root, row, "noisy", cfg)


def read_audio_file(path, cfg=CFG, data_root=None):
    """Read exactly one file; relative paths use data_root, or the working folder."""
    path = Path(path).expanduser()
    if not path.is_absolute() and data_root is not None:
        path = Path(data_root) / path
    path = path.resolve()
    x, sample_rate = sf.read(path, dtype="float64")
    if sample_rate != cfg.sample_rate:
        raise ValueError(f"Expected {cfg.sample_rate} Hz, got {sample_rate} Hz: {path}")
    return waveform(x), path




def select_examples(manifest, noise_category=None, noise_code=None,
                    target_snr_db=None, noise_difficulty=None):
    """Select metadata rows only. None means all values; no waveform is processed."""
    selected = manifest.copy()
    for column, value in (("noise_category", noise_category), ("noise_code", noise_code),
                           ("noise_difficulty", noise_difficulty)):
        if value is not None:
            selected = selected[selected[column].eq(value)]
    if target_snr_db is not None:
        selected = selected[np.isclose(selected.target_snr_db, float(target_snr_db))]
    return selected.sort_values("segment_id").reset_index(drop=True)


In [36]:
# Train / Validation manifest 로딩
train_df = load_manifest(DATA_ROOT, "train")
validation_df = load_manifest(DATA_ROOT, "validation")

print(f"Train: {len(train_df)} files")
print(f"Validation: {len(validation_df)} files")
display(validation_df.head())


Train: 1296 files
Validation: 164 files


,segment_id,clean_path,noisy_path,sample_rate_hz,num_samples,noise_category,noise_collection,noise_code,noise_difficulty,target_snr_db,split
0,speaker015_utt01_seg0001,validation/clean/speaker015_utt01_seg0001.wav,validation/noisy/speaker015_utt01_seg0001_nois...,16000,48000,aircraft_cockpit,NOISEX-92,f16,easy,10.0,validation
1,speaker015_utt01_seg0002,validation/clean/speaker015_utt01_seg0002.wav,validation/noisy/speaker015_utt01_seg0002_nois...,16000,48000,machine,Nonspeech,N026,moderate,15.0,validation
2,speaker015_utt01_seg0003,validation/clean/speaker015_utt01_seg0003.wav,validation/noisy/speaker015_utt01_seg0003_nois...,16000,48000,alarm_siren,Nonspeech,N035,easy,5.0,validation
3,speaker015_utt01_seg0004,validation/clean/speaker015_utt01_seg0004.wav,validation/noisy/speaker015_utt01_seg0004_nois...,16000,48000,alarm_siren,Nonspeech,N037,hard,15.0,validation
4,speaker015_utt01_seg0005,validation/clean/speaker015_utt01_seg0005.wav,validation/noisy/speaker015_utt01_seg0005_nois...,16000,48000,traffic_vehicle,Nonspeech,N045,easy,15.0,validation


In [37]:
def basic_filter(noisy, kind="combined", cfg=CFG):
    """Butterworth SOS, forward/backward filtering (offline, zero phase)."""
    y = waveform(noisy).copy()
    stages = cfg.basic_stages if kind == "combined" else (kind,)
    cutoffs = {"lowpass": cfg.lowpass_hz, "highpass": cfg.highpass_hz,
               "bandstop": cfg.bandstop_hz}
    for stage in stages:
        if stage not in cutoffs:
            raise ValueError(f"Unknown filter: {stage}")
        sos = signal.butter(cfg.filter_order, cutoffs[stage], btype=stage,
                            fs=cfg.sample_rate, output="sos")
        # The usual padding, bounded for very short input signals.
        ntaps = 2 * len(sos) + 1 - min((sos[:, 2] == 0).sum(), (sos[:, 5] == 0).sum())
        y = signal.sosfiltfilt(sos, y, padlen=min(3 * int(ntaps), len(y) - 1))
    return y


In [38]:
def stft_audio(x, cfg=CFG):
    x = waveform(x)
    if x.size < cfg.n_fft:
        x = np.pad(x, (0, cfg.n_fft - x.size))
    return signal.stft(x, fs=cfg.sample_rate, window="hann", nperseg=cfg.n_fft,
                       noverlap=cfg.n_fft - cfg.hop, nfft=cfg.n_fft,
                       boundary="zeros", padded=True)


def inverse_stft(z, length, cfg=CFG):
    _, y = signal.istft(z, fs=cfg.sample_rate, window="hann", nperseg=cfg.n_fft,
                        noverlap=cfg.n_fft - cfg.hop, nfft=cfg.n_fft,
                        input_onesided=True, boundary=True)
    if len(y) < length:
        raise ValueError("Inverse STFT produced insufficient samples.")
    return waveform(y[:length])


def estimate_noise_power(z, times, length, cfg=CFG):
    """Average the lowest-energy complete frames; no clean/reference access."""
    power = np.abs(z) ** 2
    # Avoid selecting artificially quiet frames introduced by boundary padding.
    centers = np.rint(times * cfg.sample_rate).astype(int)
    full = np.flatnonzero((centers >= cfg.n_fft // 2) &
                          (centers + cfg.n_fft // 2 <= length))
    if not len(full):
        full = np.arange(power.shape[1])
    energy = power[:, full].mean(axis=0)
    count = max(1, int(np.ceil(cfg.noise_fraction * len(full))))
    selected = full[np.argsort(energy, kind="stable")[:count]]
    estimate = power[:, selected].mean(axis=1)
    return np.maximum(uniform_filter1d(estimate, size=3, mode="nearest"), 1e-18)


In [39]:
def spectral_subtraction(noisy, cfg=CFG):
    x = waveform(noisy)
    _, times, z = stft_audio(x, cfg)
    power = np.abs(z) ** 2
    noise = estimate_noise_power(z, times, len(x), cfg)[:, None]
    # Power subtraction; keep the observed phase and bound gain by one.
    target = np.maximum(power - cfg.subtraction_alpha * noise,
                        cfg.subtraction_floor * power)
    gain = np.sqrt(np.clip(target / np.maximum(power, 1e-18), 0, 1))
    return inverse_stft(gain * z, len(x), cfg)


In [40]:
def wiener_filter(noisy, cfg=CFG):
    x = waveform(noisy)
    _, times, z = stft_audio(x, cfg)
    power = np.abs(z) ** 2
    noise = estimate_noise_power(z, times, len(x), cfg)
    posterior_snr = power / noise[:, None]
    enhanced = np.empty_like(z)
    previous_clean_power = np.zeros_like(noise)
    for frame in range(z.shape[1]):
        instantaneous = np.maximum(posterior_snr[:, frame] - 1.0, 0.0)
        if frame == 0:
            prior_snr = instantaneous
        else:
            prior_snr = (cfg.wiener_smoothing * previous_clean_power / noise
                         + (1 - cfg.wiener_smoothing) * instantaneous)
        gain = np.clip(prior_snr / (1 + prior_snr), cfg.wiener_gain_floor, 1)
        enhanced[:, frame] = gain * z[:, frame]
        previous_clean_power = np.abs(enhanced[:, frame]) ** 2
    return inverse_stft(enhanced, len(x), cfg)


def enhance(noisy, cfg=CFG, include_basic_variants=False):
    outputs = {"basic_filter": basic_filter(noisy, cfg=cfg),
               "spectral_subtraction": spectral_subtraction(noisy, cfg),
               "wiener_filter": wiener_filter(noisy, cfg)}
    if include_basic_variants:
        for kind in ("lowpass", "highpass", "bandstop"):
            outputs[kind] = basic_filter(noisy, kind, cfg)
    for name, output in outputs.items():
        if waveform(output).shape != np.asarray(noisy).shape:
            raise ValueError(f"Invalid output shape: {name}")
    return outputs


In [41]:
def infer_file(noisy_path, cfg=CFG, data_root=None, include_basic_variants=True):
    """Infer one noisy file, without plotting, a clean reference or any batch loop."""
    noisy, _ = read_audio_file(noisy_path, cfg, data_root)
    return {"noisy_input": noisy, **enhance(noisy, cfg, include_basic_variants)}


def show_example(noisy_path, clean_path=None, cfg=CFG, data_root=None,
                 include_basic_variants=True, verbose=True):
    """Infer and display one input file. clean_path is optional and comparison-only."""
    import matplotlib.pyplot as plt
    from IPython.display import Audio, Markdown, display

    from time import perf_counter
    started = perf_counter()
    if verbose:
        print("[1/3] Reading audio and running filters...", flush=True)
    outputs = infer_file(noisy_path, cfg, data_root, include_basic_variants)
    noisy = outputs["noisy_input"]
    if clean_path is not None:
        clean, _ = read_audio_file(clean_path, cfg, data_root)
        if clean.shape != noisy.shape:
            raise ValueError("Clean and noisy files must have the same length.")
        outputs = {"clean_reference": clean, **outputs}
    if verbose:
        print(f"[2/3] Building spectrograms ({perf_counter() - started:.2f}s)...", flush=True)
    label = Path(noisy_path).name
    reference_magnitude = max(float(np.abs(stft_audio(noisy, cfg)[2]).max()), 1e-12)
    fig, axes = plt.subplots(len(outputs), 1, figsize=(12, 2.15 * len(outputs)),
                             sharex=True, sharey=True, constrained_layout=True)
    for ax, (name, y) in zip(axes, outputs.items()):
        frequencies, times, z = stft_audio(y, cfg)
        db = 20 * np.log10(np.maximum(np.abs(z), 1e-12) / reference_magnitude)
        mesh = ax.pcolormesh(times, frequencies / 1000, db, shading="auto",
                             cmap="magma", vmin=-80, vmax=0)
        ax.set(title=name, ylabel="Frequency (kHz)")
    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(f"{label} | shared dB reference: noisy peak")
    fig.colorbar(mesh, ax=list(axes), label="dB relative to noisy STFT peak")
    # Publish directly: GUI backends can block in plt.show().
    try:
        display(fig)
    finally:
        plt.close(fig)
    if verbose:
        print(f"[3/3] Building audio players ({perf_counter() - started:.2f}s)...", flush=True)
    # A single playback gain preserves level differences across all methods.
    playback_gain = 0.99 / max(1.0, max(float(np.abs(y).max()) for y in outputs.values()))
    display(Markdown(f"**{label}** — 공통 재생 gain: {playback_gain:.4f}"))
    for name, y in outputs.items():
        display(Markdown(f"**{name}**"))
        display(Audio(y * playback_gain, rate=cfg.sample_rate, normalize=False))
    if verbose:
        print(f"Done ({perf_counter() - started:.2f}s).", flush=True)
    return outputs


In [42]:
def snr_db(clean, estimate):
    clean, estimate = waveform(clean), waveform(estimate)
    if clean.shape != estimate.shape:
        raise ValueError("SNR requires equal lengths.")
    energy = np.sum(clean ** 2)
    if energy <= 0:
        raise ValueError("SNR is undefined for a silent reference.")
    error = np.sum((clean - estimate) ** 2)
    return float(np.inf if error == 0 else 10 * np.log10(energy / error))


def pesq_score(clean, estimate, sample_rate=16000):
    from pesq import pesq, PesqError
    try:
        value = float(pesq(sample_rate, clean, estimate, mode="wb",
                           on_error=PesqError.RAISE_EXCEPTION))
        if not np.isfinite(value) or value < 0:
            raise ValueError(f"Invalid PESQ return value: {value}")
        return value, ""
    except (PesqError, ValueError) as error:
        return np.nan, f"{type(error).__name__}: {error}"


def summarize_scores(detail, group_columns=("method",)):
    summary = detail.groupby(list(group_columns), dropna=False, sort=False).agg(
        n_total=("segment_id", "size"),
        snr_mean_db=("snr_db", "mean"), snr_std_db=("snr_db", "std"),
        delta_snr_mean_db=("delta_snr_db", "mean"),
        pesq_mean=("pesq", "mean"), pesq_std=("pesq", "std"),
        pesq_valid=("pesq", "count"),
        delta_pesq_mean=("delta_pesq", "mean"),
        delta_pesq_valid=("delta_pesq", "count"),
    ).reset_index()
    summary["pesq_failed"] = summary.n_total - summary.pesq_valid
    return summary


def evaluate_dataset(root, manifest, output_dir, cfg=CFG, include_basic_variants=True,
                     split="validation"):
    # Fail before processing if PESQ is not installed; never silently omit the metric.
    import pesq  # noqa: F401
    if split not in ("train", "validation"):
        raise ValueError("split must be train or validation")
    if "split" in manifest and not manifest["split"].eq(split).all():
        raise ValueError(f"Evaluation expects the {split} split.")
    if manifest.empty or manifest.segment_id.duplicated().any():
        raise ValueError("Empty or duplicate evaluation manifest.")
    rows = []
    for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc=f"Evaluate {split}"):
        clean, noisy = read_pair(root, row, cfg)
        outputs = {"noisy_input": noisy, **enhance(noisy, cfg, include_basic_variants)}
        # Evaluate the same valid region for all methods; infer on the full waveform.
        valid = int(row.get("valid_samples", len(clean)))
        if not 0 < valid <= len(clean):
            raise ValueError(f"Invalid valid_samples: {row['segment_id']}")
        clean = clean[:valid]
        input_snr = snr_db(clean, noisy[:valid])
        input_pesq, input_error = pesq_score(clean, noisy[:valid], cfg.sample_rate)
        for method, y in outputs.items():
            score_snr = snr_db(clean, y[:valid])
            score_pesq, error = ((input_pesq, input_error) if method == "noisy_input"
                                 else pesq_score(clean, y[:valid], cfg.sample_rate))
            rows.append({"split": split, "segment_id": row["segment_id"], "method": method,
                         "noise_category": row.get("noise_category", "unknown"),
                         "noise_difficulty": row.get("noise_difficulty", "unknown"),
                         "target_snr_db": row.get("target_snr_db", np.nan),
                         "snr_db": score_snr, "delta_snr_db": score_snr - input_snr,
                         "pesq": score_pesq, "delta_pesq": score_pesq - input_pesq,
                         "pesq_error": error})
    detail = pd.DataFrame(rows)
    summary = summarize_scores(detail)
    # Common successful subset for fair PESQ comparisons if a method has failures.
    complete = detail.groupby("segment_id").pesq.apply(lambda x: x.notna().all())
    common = detail[detail.segment_id.isin(complete[complete].index)]
    common_summary = summarize_scores(common)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    detail.to_csv(output_dir / f"{split}_per_file.csv", index=False, encoding="utf-8-sig")
    summary.to_csv(output_dir / f"{split}_summary.csv", index=False, encoding="utf-8-sig")
    common_summary.to_csv(output_dir / f"{split}_common_subset.csv", index=False,
                          encoding="utf-8-sig")
    for group in ("noise_category", "noise_difficulty", "target_snr_db"):
        summarize_scores(detail, ("method", group)).to_csv(
            output_dir / f"{split}_by_{group}.csv", index=False, encoding="utf-8-sig")
    versions = {name: importlib.metadata.version(name) for name in
                ("numpy", "scipy", "soundfile", "pandas", "pesq")}
    metadata = {"config": asdict(cfg), "python": platform.python_version(),
                "versions": versions, "split": split, "evaluated_files": len(manifest),
                "methods": detail.method.unique().tolist(), "pesq_mode": "wb",
                "common_pesq_files": int(complete.sum()),
                "snr": "10*log10(sum(clean**2)/sum((clean-output)**2)); no level alignment",
                "aggregation": "unweighted arithmetic mean across files"}
    (output_dir / "run_config.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    if summary.pesq_failed.sum():
        print("PESQ failures detected: inspect pesq_error and compare the common-subset table.")
    return detail, summary




def test_all_samples(root, split="validation", output_dir=None, cfg=CFG,
                     include_basic_variants=True):
    """Reload and evaluate EVERY file of a split, independently of example filters.

    split='all' evaluates train and validation and returns their combined tables.
    No plots or playback widgets are created here.
    """
    if split not in ("train", "validation", "all"):
        raise ValueError("split must be train, validation or all")
    root = Path(root).resolve()
    output_dir = Path(output_dir) if output_dir is not None else root / "outputs" / "evaluation"
    splits = ("train", "validation") if split == "all" else (split,)
    details = []
    for name in splits:
        full_manifest = load_manifest(root, name)
        destination = output_dir / name if split == "all" else output_dir
        detail, summary = evaluate_dataset(root, full_manifest, destination, cfg,
                                            include_basic_variants, split=name)
        if not detail.groupby("method").size().eq(len(full_manifest)).all():
            raise RuntimeError("Not every manifest entry was evaluated.")
        details.append(detail)
    if split != "all":
        return details[0], summary
    combined = pd.concat(details, ignore_index=True)
    summary = summarize_scores(combined, ("split", "method"))
    combined.to_csv(output_dir / "all_per_file.csv", index=False, encoding="utf-8-sig")
    summary.to_csv(output_dir / "all_summary.csv", index=False, encoding="utf-8-sig")
    return combined, summary


In [43]:
required_functions = [
    "waveform", "stft_audio", "inverse_stft", "estimate_noise_power",
    "basic_filter", "spectral_subtraction", "wiener_filter", "enhance",
    "snr_db", "pesq_score", "evaluate_dataset", "test_all_samples",
]

for name in required_functions:
    print(f"[{'OK' if callable(globals().get(name)) else 'MISSING'}] {name}")

assert all(callable(globals().get(name)) for name in required_functions)

# 알고리즘은 Noisy 파형만 입력으로 받습니다.
sample_noisy = read_audio(DATA_ROOT, validation_df.iloc[0], "noisy", CFG)
sample_outputs = enhance(sample_noisy, CFG)

print("\nNoisy shape:", sample_noisy.shape)
for method, result in sample_outputs.items():
    print(f"{method}: shape={result.shape}, finite={np.isfinite(result).all()}")
    assert result.shape == sample_noisy.shape
    assert np.isfinite(result).all()

print("\nBaseline smoke test passed.")


[OK] waveform
[OK] stft_audio
[OK] inverse_stft
[OK] estimate_noise_power
[OK] basic_filter
[OK] spectral_subtraction
[OK] wiener_filter
[OK] enhance
[OK] snr_db
[OK] pesq_score
[OK] evaluate_dataset
[OK] test_all_samples

Noisy shape: (48000,)
basic_filter: shape=(48000,), finite=True
spectral_subtraction: shape=(48000,), finite=True
wiener_filter: shape=(48000,), finite=True

Baseline smoke test passed.


In [44]:
EVALUATION_DIR = BASE_DIR / "experiments" / "baseline" / "evaluation"

summary_path = EVALUATION_DIR / "validation_summary.csv"
category_path = EVALUATION_DIR / "validation_by_noise_category.csv"
config_path = EVALUATION_DIR / "run_config.json"

if summary_path.is_file():
    baseline_summary = pd.read_csv(summary_path)
    display(baseline_summary.round(4))
else:
    print("기존 validation_summary.csv를 찾지 못했습니다:", summary_path)

if category_path.is_file():
    category_df = pd.read_csv(category_path)
    methods = ["noisy_input", "spectral_subtraction", "wiener_filter"]
    focus = category_df[category_df["method"].isin(methods)].copy()

    for metric in [
        "n_total", "pesq_mean", "delta_pesq_mean",
        "snr_mean_db", "delta_snr_mean_db"
    ]:
        if metric in focus.columns:
            print(f"\n===== {metric} =====")
            display(
                focus.pivot(
                    index="noise_category",
                    columns="method",
                    values=metric
                ).round(3)
            )
else:
    print("기존 validation_by_noise_category.csv를 찾지 못했습니다:", category_path)

if config_path.is_file():
    with config_path.open("r", encoding="utf-8") as f:
        previous_run_config = json.load(f)
    print("\n기존 Baseline 실행 설정:")
    print(json.dumps(previous_run_config, indent=2, ensure_ascii=False))
else:
    print("기존 run_config.json을 찾지 못했습니다:", config_path)


,method,n_total,snr_mean_db,snr_std_db,delta_snr_mean_db,pesq_mean,pesq_std,pesq_valid,delta_pesq_mean,delta_pesq_valid,pesq_failed
0,noisy_input,164,9.6341,4.8461,0.0000,1.3947,0.3526,164,0.0000,164,0
1,basic_filter,164,9.0697,4.0345,-0.5645,1.4486,0.3386,164,0.0539,164,0
2,spectral_subtraction,164,12.5683,4.5321,2.9341,1.5376,0.4253,164,0.1429,164,0
3,wiener_filter,164,13.1196,3.8668,3.4854,1.5984,0.4587,164,0.2037,164,0
4,lowpass,164,9.6413,4.8041,0.0072,1.4225,0.3206,164,0.0278,164,0
5,highpass,164,9.4732,4.6148,-0.1609,1.4291,0.3941,164,0.0344,164,0
6,bandstop,164,9.2538,4.5805,-0.3803,1.3887,0.3439,164,-0.0059,164,0



===== n_total =====


method,noisy_input,spectral_subtraction,wiener_filter
noise_category,,,
aircraft_cockpit,14,14,14
alarm_siren,23,23,23
animal,2,2,2
bell,3,3,3
broadband_noise,10,10,10
clap,1,1,1
click,1,1,1
destroyer_engine,5,5,5
destroyer_operations,5,5,5



===== pesq_mean =====


method,noisy_input,spectral_subtraction,wiener_filter
noise_category,,,
aircraft_cockpit,1.173,1.374,1.701
alarm_siren,1.311,1.422,1.479
animal,1.423,1.411,1.437
bell,1.675,1.568,1.417
broadband_noise,1.115,1.277,1.528
clap,1.378,1.313,1.284
click,2.129,1.849,1.437
destroyer_engine,1.248,1.469,1.575
destroyer_operations,1.286,1.609,1.613



===== delta_pesq_mean =====


... (2 outputs left out)

method,noisy_input,spectral_subtraction,wiener_filter
noise_category,,,
aircraft_cockpit,9.643,13.215,13.563
alarm_siren,8.043,11.886,12.403
animal,12.500,13.624,15.692
bell,13.333,14.637,14.512
broadband_noise,7.500,11.866,12.958
clap,15.000,15.023,15.118
click,15.000,15.036,12.231
destroyer_engine,8.000,12.279,13.429
destroyer_operations,9.000,11.768,11.202



===== delta_snr_mean_db =====


method,noisy_input,spectral_subtraction,wiener_filter
noise_category,,,
aircraft_cockpit,0.0,3.572,3.920
alarm_siren,0.0,3.842,4.360
animal,0.0,1.124,3.192
bell,0.0,1.303,1.178
broadband_noise,0.0,4.366,5.458
clap,0.0,0.023,0.118
click,0.0,0.036,-2.769
destroyer_engine,0.0,4.279,5.429
destroyer_operations,0.0,2.768,2.202



기존 Baseline 실행 설정:
{
  "config": {
    "sample_rate": 16000,
    "n_fft": 512,
    "hop": 128,
    "filter_order": 4,
    "lowpass_hz": 3500.0,
    "highpass_hz": 100.0,
    "bandstop_hz": [
      950.0,
      1050.0
    ],
    "basic_stages": [
      "highpass",
      "lowpass",
      "bandstop"
    ],
    "noise_fraction": 0.2,
    "subtraction_alpha": 1.5,
    "subtraction_floor": 0.02,
    "wiener_smoothing": 0.98,
    "wiener_gain_floor": 0.05
  },
  "python": "3.13.18",
  "versions": {
    "numpy": "2.4.6",
    "scipy": "1.16.3",
    "soundfile": "0.14.0",
    "pandas": "2.2.3",
    "pesq": "0.0.4"
  },
  "split": "validation",
  "evaluated_files": 164,
  "methods": [
    "noisy_input",
    "basic_filter",
    "spectral_subtraction",
    "wiener_filter",
    "lowpass",
    "highpass",
    "bandstop"
  ],
  "pesq_mode": "wb",
  "common_pesq_files": 164,
  "snr": "10*log10(sum(clean**2)/sum((clean-output)**2)); no level alignment",
  "aggregation": "unweighted arithmetic mean acro

In [45]:
RUN_VALIDATION_EVALUATION = False

if RUN_VALIDATION_EVALUATION:
    rerun_dir = BASE_DIR / "experiments" / "baseline" / "evaluation_rerun"
    rerun_detail, rerun_summary = test_all_samples(
        DATA_ROOT,
        split="validation",
        output_dir=rerun_dir,
        cfg=CFG,
        include_basic_variants=True,
    )
    print("재평가 완료:", rerun_dir)
    display(rerun_summary.round(4))
else:
    print("Validation 재평가를 건너뜁니다. 기존 Baseline 결과는 변경되지 않았습니다.")


Validation 재평가를 건너뜁니다. 기존 Baseline 결과는 변경되지 않았습니다.
